# Week 5 · 01 From a formula to NumPy: vectorized code on purpose

You can already write the loop. This notebook teaches the step after it: from a formula with indices, or from your own loop, to the one-line NumPy version, by a method with six fixed steps. It starts with the exact lines you couldn't read in Week 4 (`e_step`, `m_step`), then uses the same method on every formula deep learning needs this week: a dense layer forward and backward, softmax, cross-entropy. Three light days: §M–4 on Wednesday, §5–8 on Thursday, §9–11 on Friday. 12 exercises, all checked.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **index**: a letter such as $i$ or $k$ in $x_i$ or $r_{ik}$ that says **which** entry. Each index becomes one axis of an array.
- **axis**: one direction of an array. Axis 0 runs down the rows, axis 1 across the columns. `shape` lists the length of each axis.
- **broadcasting**: how NumPy combines arrays of different shapes: line the shapes up from the right; a length-1 axis is stretched (copied) to match the other array.
- **reduction**: an operation that removes an axis: `sum`, `mean`, `max`, `argmax` along `axis=...`.
- **vectorized code**: code that works on whole arrays at once, with no Python loop over the entries.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "01")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** The method this notebook teaches. Each step answers one question: what are the indices, which axis is each, where does each input need a `None`, how are they combined, which index is summed away, and is the result right?

In [ ]:
fig_method_map()

| Section | What | Why it's here |
|---|---|---|
| 0 | recap, and softmax from memory | the NumPy you need today, closed book |
| 1 | why vectorize | speed, and reading PyTorch code: the reasons for learning a method at all |
| 2 | the method, as an algorithm card | six steps you can apply to any formula |
| 3 | your `e_step`: from your loop to one line | the line you couldn't read in Week 4, rebuilt step by step |
| 4 | summing the index that disappears: the M-step | the other Week 4 lines, such as `(R*x[:,None]).sum(axis=0)/Nk` |
| 5 | the translation table: loop → NumPy | a dictionary for turning any loop you write into array code |
| 6 | matrix multiplication as a shared index | when a sum over a shared index is just `@` |
| 7–8 | a dense layer, forward and backward | the core of every neural network (`02`, `03`), by the same method |
| 9 | softmax and cross-entropy, row by row | the output layer and loss of every classifier this week |
| 10 | einsum (optional) | the method written as a single string; it shows up in PyTorch code |
| 11 | the recipe | the method on one card, and all 12 checks |

**How it connects.** Every deep-learning formula this week has indices: sample $i$, input feature $j$, unit $k$, class $c$. The method turns each of them into one line of NumPy, and PyTorch code is written in exactly the same style, so learning to write it also teaches you to read it.

## 0. Recap first · 10 min, closed book

**Quick check:** `np.array([1, 2, 3])[:, None].shape` is ...?

<details><summary>Answer</summary>

`(3, 1)`: `None` inserts a new axis of length 1 at that position, so the 3 numbers become a column. [Week 3 · 02 §7 ↗](../week3/02_knn_and_numpy_workout.ipynb)

</details>

**Quick check:** Arrays of shapes `(6, 1)` and `(1, 2)` are combined with `-`. What's the result's shape, and what's the rule?

<details><summary>Answer</summary>

`(6, 2)`. Line the shapes up from the right; at each position the two lengths must be equal or one of them must be 1, and the 1 is stretched to the other length. [Week 3 · 02 §7 ↗](../week3/02_knn_and_numpy_workout.ipynb)

</details>

**Quick check:** `R` has shape `(6, 2)`. Shapes of `R.sum(axis=0)`, `R.sum(axis=1)` and `R.sum(axis=1, keepdims=True)`?

<details><summary>Answer</summary>

`(2,)`, `(6,)` and `(6, 1)`. The axis you sum along disappears, unless `keepdims=True` keeps it with length 1. [Week 4 · 00 §C ↗](../week4/00_week3_review.ipynb)

</details>

**Quick check:** `A` is `(n, d)` and `B` is `(d, h)`. Shape of `A @ B`, and what must match?

<details><summary>Answer</summary>

`(n, h)`. The inner lengths, both `d`, must be equal; they disappear in the product.

</details>

**Quick check:** `P` has shape `(3, 4)` and `y = np.array([2, 0, 1])`. Which entries does `P[np.arange(3), y]` pick?

<details><summary>Answer</summary>

`P[0, 2]`, `P[1, 0]` and `P[2, 1]`: the row numbers `[0, 1, 2]` and the column numbers `y` are paired up position by position. One entry per row, shape `(3,)`. [Week 3 · 02 §8 ↗](../week3/02_knn_and_numpy_workout.ipynb)

</details>

**Exercise 1 · Softmax of each row, from memory.** $p_{ic} = \dfrac{e^{z_{ic}}}{\sum_{c'} e^{z_{ic'}}}$ for every row $i$ of `Z` `(n, C)`. Subtract each row's maximum first (it doesn't change the result, and it stops `np.exp` from overflowing: the check includes a row with 1000 in it). **Shape `(n, C)`**, each row sums to 1.

**Example**: what goes in, and what should come out:
```python
Z = np.array([[1.0, 2.0, 3.0],
              [1.0, 1.0, 1.0]])     # (2, 3), float
softmax_rows(Z)
# → [[0.09  0.245 0.665]
#    [0.333 0.333 0.333]]
```

In [ ]:
def softmax_rows(Z):
    pass  # your code

In [ ]:
check(1)

## 1. Why vectorize at all

**The question this section answers:** your loop versions were correct, so why learn another way to write the same thing? Two reasons.

**Speed.** A Python loop runs one entry at a time, and each step goes through the Python interpreter. A NumPy operation runs the whole loop in compiled C. Below: the Gaussian density from Week 4 (run it, everything after uses it), then your E-step loop and the one-line version on 100,000 points.

In [ ]:
def gaussian_pdf(x, mu, var):          # Week 4 · 04 exercise 3: works on numbers AND on whole arrays
    return np.exp(-(x - mu) ** 2 / (2 * var)) / np.sqrt(2 * np.pi * var)

def e_step_loop(x, mu, var, pi):        # your Week 4 version, unchanged
    l1 = []
    for i in range(len(x)):
        l2 = []
        for k in range(len(pi)):
            l2.append(pi[k] * gaussian_pdf(x[i], mu[k], var[k]))
        l1.append(l2)
    m = np.array(l1)
    return m / m.sum(axis=1, keepdims=True)

In [ ]:
import time
x_big = np.random.default_rng(0).normal(size=100_000)
mu, var, pi = np.array([2.0, 5.0]), np.array([4.0, 4.0]), np.array([0.5, 0.5])

t0 = time.perf_counter(); R1 = e_step_loop(x_big, mu, var, pi); t1 = time.perf_counter()
m = pi[None, :] * gaussian_pdf(x_big[:, None], mu[None, :], var[None, :]); R2 = m / m.sum(axis=1, keepdims=True); t2 = time.perf_counter()
print(f"loop: {t1 - t0:.3f} s   one line: {t2 - t1:.4f} s   about {(t1 - t0) / (t2 - t1):.0f}x faster   same answer: {np.allclose(R1, R2)}")

Your numbers will differ a little, but the one-line version is typically 30–100 times faster. On a real dataset that's seconds instead of minutes, and inside a training loop that runs thousands of times, it decides whether training is possible at all.

**Reading.** Every PyTorch model is written this way: a layer is `x @ W.T + b` on a whole batch, never a loop over samples. The lab exam asks you to read such code. If you can produce a line like `pi[None, :] * gaussian_pdf(x[:, None], ...)` yourself, you can read anything written in that style.

**So the loop isn't wrong; it's a stepping stone.** Write it whenever you need to, then translate it with the method below. After enough translations you'll skip the loop.

## 2. The method, as an algorithm card

**Goal.** One NumPy expression that computes a formula for **all** values of its indices at once.

**Steps.**

- **Formula for one entry.** Write what a single entry is, with every index visible. Example: $m_{ik} = \pi_k\, \mathcal{N}(x_i \mid \mu_k, \sigma_k^2)$.
- **Anatomy.** For every symbol, list its indices; that gives its array and shape. Give each index of the **result** an axis, in order: $m_{ik}$ has $i$ then $k$, so $i$ → axis 0, $k$ → axis 1, result shape $(n, K)$.
- **Line up.** Index each input so its indices sit on their axes, with `None` (a new axis of length 1) wherever it lacks one. $x_i$ has $i$ but not $k$ → `x[:, None]`, shape $(n, 1)$; $\mu_k$ has $k$ but not $i$ → `mu[None, :]`, shape $(1, K)$.
- **Combine** with elementwise operations (`+ - * /`, `**`, `np.exp`, any function built from them). Broadcasting fills every combination of indices: this is your nested loop.
- **Reduce** every index that appears inside the formula but not in the result: $\sum_i$ → `.sum(axis=<i's axis>)`. Likewise `max`, `mean`, `argmax`.
- **Check.** The result's shape, and one entry worked out by hand against the formula.

**Why it works.** An elementwise operation on shapes $(n, 1)$ and $(1, K)$ produces an $(n, K)$ array whose entry $(i, k)$ is the operation applied to entry $i$ of the first and entry $k$ of the second: exactly what `for i: for k:` computes. And `.sum(axis=0)` is exactly `total += ...` over the index on axis 0. So steps 3–5 are your loops, rewritten without the loop syntax.

**Caveats.** (1) **Memory**: the combined array holds one number per combination of indices. Distances between $10^5$ points and $10^3$ centroids in 100 dimensions would be a $(10^5, 10^3, 100)$ array, $10^{10}$ numbers (80 GB). Then use matrix multiplication (§6) or loop over one index. (2) **Losing track of axes**: write the shape as a comment at the end of every line, `# (n, K)`. Every solution in this notebook does.

**Cost.** The same arithmetic as the loop, done in compiled code; the speed-up comes only from skipping the interpreter.

**Example.** The next section applies all six steps to your E-step.

## 3. Your `e_step`: from your loop to one line

> ↩ **Recap** · [Week 4 · 04 exercise 4 (E-step) ↗](../week4/04_hierarchical_dbscan_gmm.ipynb)
>
> Data `x = [1, 2, 3, 6, 7, 8]` (6 points), two bumps with $\mu = (2, 5)$, $\sigma^2 = (4, 4)$, $\pi = (0.5, 0.5)$. Your loop (`e_step_loop` in §1) builds the table one entry at a time, then divides each row by its sum.

**Step 1: one entry.** The innermost line of your loop, `pi[k] * gaussian_pdf(x[i], mu[k], var[k])`, is the formula for one entry:

$$
m_{ik} = \pi_k\, \mathcal{N}(x_i \mid \mu_k, \sigma_k^2), \qquad r_{ik} = \frac{m_{ik}}{\sum_{j} m_{ij}}
$$

**Step 2: anatomy.** Each symbol, its indices, and the array that holds it:

| Symbol | Indices | Array | Shape | What it is |
|---|---|---|---|---|
| $x_i$ | $i$ | `x` | `(n,)` = `(6,)` | the data points |
| $\mu_k,\ \sigma_k^2,\ \pi_k$ | $k$ | `mu`, `var`, `pi` | `(K,)` = `(2,)` | each bump's centre, spread and weight |
| $m_{ik}$ | $i, k$ | `m` | `(n, K)` | point $i$ under bump $k$, not yet normalized |
| $r_{ik}$ | $i, k$ | `R` | `(n, K)` | the share of point $i$ that bump $k$ explains; rows sum to 1 |

The result has indices $i, k$, so $i$ → axis 0 and $k$ → axis 1: one **row per point**, one **column per bump**.

**Step 3: line up.** `x` carries $i$ only, so it goes on axis 0 with an empty axis 1: `x[:, None]`, shape `(6, 1)`. `mu`, `var`, `pi` carry $k$ only, so they go on axis 1 with an empty axis 0: `mu[None, :]`, shape `(1, 2)`.

🖼 **Picture.** What `x[:, None] - mu[None, :]` does. The column `x[:, None]` is copied across the 2 columns, the row `mu[None, :]` is copied down the 6 rows, and then they're subtracted entry by entry. Yellow: entry $(2, 1)$, i.e. $x_2 - \mu_1 = 3 - 5 = -2$.

In [ ]:
fig_estep_broadcast()

**Step 4: combine.** `gaussian_pdf` uses only elementwise operations (`-`, `**`, `/`, `np.exp`, `np.sqrt`), so it works unchanged on whole arrays: given shapes `(6, 1)`, `(1, 2)`, `(1, 2)` it returns `(6, 2)`, every $\mathcal{N}(x_i \mid \mu_k, \sigma_k^2)$ at once. Then `pi[None, :] *` multiplies column $k$ by $\pi_k$. **Step 5** has nothing to reduce for $m$; the normalization sums over $j$ (a bump index) inside each row, so it's `m.sum(axis=1, keepdims=True)`, shape `(6, 1)`, and each row is divided by its own sum.

🖼 **Picture.** The rest of the E-step on the same data. Row 2 (the point $x = 3$) is followed through every array.

In [ ]:
fig_estep_rest()

Every intermediate result, line by line:

| Code | Shape | Entry `[i, k]` is |
|---|---|---|
| `x[:, None]` | `(6, 1)` | $x_i$ (same in every column) |
| `mu[None, :]` | `(1, 2)` | $\mu_k$ (same in every row) |
| `x[:, None] - mu[None, :]` | `(6, 2)` | $x_i - \mu_k$ |
| `gaussian_pdf(x[:, None], mu[None, :], var[None, :])` | `(6, 2)` | $\mathcal{N}(x_i \mid \mu_k, \sigma_k^2)$ |
| `m = pi[None, :] * (...)` | `(6, 2)` | $m_{ik} = \pi_k\, \mathcal{N}(x_i \mid \mu_k, \sigma_k^2)$ |
| `m.sum(axis=1, keepdims=True)` | `(6, 1)` | $\sum_j m_{ij}$ (one sum per row) |
| `m / m.sum(axis=1, keepdims=True)` | `(6, 2)` | $r_{ik}$ |

**Step 6: check.** The shape is `(6, 2)`, every row sums to 1, entry $(2, 0)$ is $0.088 / 0.1485 = 0.593$ as in the picture, and the result equals your loop's:

In [ ]:
x = np.array([1, 2, 3, 6, 7, 8.0])
mu, var, pi = np.array([2.0, 5.0]), np.array([4.0, 4.0]), np.array([0.5, 0.5])
m = pi[None, :] * gaussian_pdf(x[:, None], mu[None, :], var[None, :])   # (6, 2)
R = m / m.sum(axis=1, keepdims=True)                                     # (6, 2)
print(R.shape, R.sum(axis=1).round(6))         # → (6, 2) [1. 1. 1. 1. 1. 1.]
print(R[:, 0].round(3))                        # → [0.867 0.755 0.593 0.133 0.068 0.033]
print(np.allclose(R, e_step_loop(x, mu, var, pi)))   # → True

**Do you need every `None`?** Broadcasting lines shapes up from the **right** and adds missing axes on the **left**. So `pi`, shape `(2,)`, already behaves like `(1, 2)`, and `pi * G` works without the `None`. But `x`, shape `(6,)`, would also behave like `(1, 6)`, which doesn't fit `(1, 2)`:

In [ ]:
try:
    x - mu                    # (6,) and (2,): lined up from the right, 6 vs 2
except ValueError as e:
    print("ValueError:", e)
# → ValueError: operands could not be broadcast together with shapes (6,) (2,)

So the rule: **an index on the last axis can do without its `None`; every other one needs it.** Writing them all is clearer, and it's what the solutions do.

**Recipe (an E-step style formula).** Data index on axis 0 (`x[:, None]`), component index on axis 1 (`mu[None, :]`), apply the elementwise function, normalize each row with `.sum(axis=1, keepdims=True)`.

**Exercise 2 · E-step, without looking.** $r_{ik} = \dfrac{\pi_k\, \mathcal{N}(x_i \mid \mu_k, \sigma_k^2)}{\sum_j \pi_j\, \mathcal{N}(x_i \mid \mu_j, \sigma_j^2)}$. `x`: `(n,)`; `mu`, `var`, `pi`: `(K,)`. Return `R`, **shape `(n, K)`**, no loops. Scroll so the code above is off screen and work from the anatomy table. Uses `gaussian_pdf` from §1.

**Example**: what goes in, and what should come out:
```python
x = np.array([1, 2, 3, 6, 7, 8.0])
R = e_step(x, np.array([2.0, 5.0]), np.array([4.0, 4.0]), np.array([0.5, 0.5]))
R.shape     # → (6, 2)
R[:, 0]     # → [0.867 0.755 0.593 0.133 0.068 0.033]
```

In [ ]:
def e_step(x, mu, var, pi):
    pass  # your code

In [ ]:
check(2)

## 4. Summing the index that disappears: the M-step

> ↩ **Recap** · [Week 4 · 04 §3 (the M-step formulas) ↗](../week4/04_hierarchical_dbscan_gmm.ipynb)
>
> Given the responsibilities $R$ `(n, K)`, each bump is refit: $N_k = \sum_i r_{ik}$, $\mu_k = \frac{1}{N_k}\sum_i r_{ik} x_i$, $\sigma_k^2 = \frac{1}{N_k}\sum_i r_{ik}(x_i - \mu_k)^2$ (around the **new** $\mu_k$), $\pi_k = \frac{N_k}{n}$.

**The question:** what does `mu = (R * x[:, None]).sum(axis=0) / Nk` do, and how would you come up with it? Start with the loop you'd write for these formulas. Notice that every formula has a $\sum_i$: in the loop, that's an inner loop over $i$ adding things up.

In [ ]:
def m_step_loop(x, R):
    n, K = R.shape
    mu, var, pi = np.zeros(K), np.zeros(K), np.zeros(K)
    for k in range(K):
        Nk, s = 0.0, 0.0
        for i in range(n):
            Nk += R[i, k]                 # N_k = sum over i of r_ik
            s += R[i, k] * x[i]           # sum over i of r_ik x_i
        mu[k] = s / Nk
        s2 = 0.0
        for i in range(n):
            s2 += R[i, k] * (x[i] - mu[k]) ** 2
        var[k] = s2 / Nk
        pi[k] = Nk / n
    return mu, var, pi

Now apply the method to $\mu_k = \frac{1}{N_k}\sum_i r_{ik} x_i$. **What's inside the sum** has indices $i$ and $k$, so it's an $(n, K)$ table with entry $(i, k) = r_{ik}\, x_i$: that's `R * x[:, None]` (`x` carries only $i$, so it gets `[:, None]`). **The sum runs over $i$**, which is axis 0, so `.sum(axis=0)`: the 6 rows collapse, one number per bump is left, shape `(K,)`. Dividing by `Nk`, also `(K,)`, divides bump by bump.

🖼 **Picture.** $\mu_k$ step by step on the Week 4 data. The column $k = 0$ (blue) is bump 0's computation: $\sum_i r_{i0} x_i = 5.69$, and $N_0 = 2.448$, so $\mu_0 = 2.324$.

In [ ]:
fig_mstep_mean()

Every M-step line, translated the same way:

| Formula | Inside the sum: indices → array | Summed over | Code | Shape |
|---|---|---|---|---|
| $N_k = \sum_i r_{ik}$ | $i, k$ → `R` | $i$ (axis 0) | `Nk = R.sum(axis=0)` | `(K,)` |
| $\mu_k = \frac{1}{N_k}\sum_i r_{ik} x_i$ | $i, k$ → `R * x[:, None]` | $i$ (axis 0) | `mu = (R * x[:, None]).sum(axis=0) / Nk` | `(K,)` |
| $\sigma_k^2 = \frac{1}{N_k}\sum_i r_{ik}(x_i - \mu_k)^2$ | $i, k$ → `R * (x[:, None] - mu[None, :]) ** 2` | $i$ (axis 0) | `var = (...).sum(axis=0) / Nk` | `(K,)` |
| $\pi_k = N_k / n$ | no sum |  | `pi = Nk / len(x)` | `(K,)` |

In loop terms: **the loop over $k$ disappears because every $k$ is a column, and each loop over $i$ that adds things up becomes `.sum(axis=0)`.** Here are the lines run, and compared with the loop:

In [ ]:
Nk = R.sum(axis=0)                                            # (2,)
mu_new = (R * x[:, None]).sum(axis=0) / Nk                    # (6, 2) -> (2,), divided by (2,)
var_new = (R * (x[:, None] - mu_new[None, :]) ** 2).sum(axis=0) / Nk
print(Nk.round(3), mu_new.round(3), var_new.round(3), (Nk / len(x)).round(3))
# → [2.448 3.552] [2.324 6.   ] [2.536 4.424] [0.408 0.592]
print(all(np.allclose(a, b) for a, b in zip((mu_new, var_new, Nk / len(x)), m_step_loop(x, R))))   # → True

> ⚠️ **Common mistake.** `R * x` without `[:, None]`: `x` `(6,)` acts as `(1, 6)` against `R` `(6, 2)`, so the last axes are 6 vs 2 and NumPy raises an error. Worse, if $n$ happened to equal $K$, it would run and silently multiply column $k$ by $x_k$: the wrong formula with no error message. The `None` is what puts $x_i$ along the rows.

**Recipe (a weighted sum over the data index).** Build the $(n, K)$ table of what's inside the sum, then `.sum(axis=0)`. The weights `R` are already $(n, K)$; anything with only $i$ gets `[:, None]`; anything with only $k$ gets `[None, :]`.

**Exercise 3 · Weighted means.** For each column $k$ of the weights `W` `(n, K)`: $\dfrac{\sum_i W_{ik}\, x_i}{\sum_i W_{ik}}$. `x` is `(n,)`. Return **shape `(K,)`**, no loops.

**Example**: what goes in, and what should come out:
```python
x3 = np.array([1.0, 2.0, 3.0])
W3 = np.array([[1.0, 0.0],
               [1.0, 1.0],
               [0.0, 1.0]])      # (3, 2): column 0 weights the first two points, column 1 the last two
weighted_means(x3, W3)    # → [1.5 2.5]
```

In [ ]:
def weighted_means(x, W):
    pass  # your code

In [ ]:
check(3)

**Exercise 4 · M-step, without looking.** Return `(mu, var, pi)`, each **shape `(K,)`**: $N_k = \sum_i r_{ik}$, $\mu_k = \frac{1}{N_k}\sum_i r_{ik} x_i$, $\sigma_k^2 = \frac{1}{N_k}\sum_i r_{ik}(x_i - \mu_k)^2$ with the **new** $\mu_k$, $\pi_k = N_k / n$. `x` `(n,)`, `R` `(n, K)`. No loops.

**Example**: what goes in, and what should come out:
```python
mu, var, pi = m_step(x, R)     # x and R from §3
mu      # → [2.324 6.   ]
var     # → [2.536 4.424]
pi      # → [0.408 0.592]
```

In [ ]:
def m_step(x, R):
    pass  # your code

In [ ]:
check(4)

## 5. The translation table: loop → NumPy

The patterns you'll meet over and over. When you've written a loop, find its line in the left column.

| Loop pattern | Formula | NumPy | Example |
|---|---|---|---|
| `out[i] = f(a[i])` | $o_i = f(a_i)$ | `f(a)` | `np.exp(z)` |
| `out[i][k] = f(a[i], b[k])` | $o_{ik} = f(a_i, b_k)$ | `f(a[:, None], b[None, :])` | E-step, distance tables |
| `s += a[i]` over all `i` | $\sum_i a_i$ | `a.sum()` | total weight |
| for each `k`: `s += A[i, k]` over `i` | $\sum_i A_{ik}$ | `A.sum(axis=0)` | $N_k$ |
| for each `i`: `s += A[i, k]` over `k` | $\sum_k A_{ik}$ | `A.sum(axis=1)` | row sums |
| `s += a[i] * b[i]` | $\sum_i a_i b_i$ | `a @ b` | a dot product |
| `out[i] = sum over k of A[i, k] * b[k]` | $\sum_k A_{ik} b_k$ | `A @ b` | scores $Xw$ |
| `count += (a[i] <= t)` | $\#\{i : a_i \le t\}$ | `(a <= t).sum()` | DBSCAN neighbours |
| `out[i] = a[i] if c[i] else b[i]` |  | `np.where(c, a, b)` | stump predictions |
| `best = k with the smallest d[i, k]` | $\arg\min_k d_{ik}$ | `d.argmin(axis=1)` | k-means assign |
| `out[i, k] = A[i, k] / sum over j of A[i, j]` | $A_{ik} / \sum_j A_{ij}$ | `A / A.sum(axis=1, keepdims=True)` | normalize rows |
| `[x[i] for i in range(n) if m[i]]` |  | `x[m]` (`m` True/False) | the points of cluster $c$ |
| `out[i] = table[idx[i]]` |  | `table[idx]` | `np.eye(C)[y]` |

**Your own example: `dbscan_core`.** Your loop was `for row in D: l.append(np.count_nonzero(row <= eps) >= min_pts)`. Find it in the table: for each row $i$, count over $j$, i.e. "for each $i$: `s += A[i, j]` over $j$" applied to the True/False table `D <= eps`. So `(D <= eps).sum(axis=1) >= min_pts`, which is exactly the line you wrote in your comment.

**One more, all six steps: pairwise squared distances.** You've used this table since kNN. $D_{ij} = \sum_d (A_{id} - B_{jd})^2$ for `A` `(m, d)` and `B` `(p, d)`.

- Formula for one entry: $D_{ij} = \sum_d (A_{id} - B_{jd})^2$; inside the sum, the indices are $i, j, d$.
- Anatomy: `A` has $(i, d)$, `B` has $(j, d)$; put $i$ on axis 0, $j$ on axis 1, $d$ on axis 2.
- Line up: `A[:, None, :]` is $(m, 1, d)$; `B[None, :, :]` is $(1, p, d)$.
- Combine: `(A[:, None, :] - B[None, :, :]) ** 2` is $(m, p, d)$; entry $(i, j, d)$ is $(A_{id} - B_{jd})^2$.
- Reduce: $d$ is summed and not in the result → `.sum(axis=2)` → $(m, p)$.
- Check: with one point in each, it's the squared distance you'd compute by hand.

**Exercise 5 · Standardize the columns.** $z_{ij} = \dfrac{x_{ij} - \bar x_j}{s_j}$, with $\bar x_j$ and $s_j$ the mean and standard deviation of column $j$ (`np.std`'s default, dividing by $n$). `X` is `(n, d)`; return **shape `(n, d)`**.

**Example**: what goes in, and what should come out:
```python
standardize(np.array([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]]))
# → [[-1.225 -1.225]
#    [ 0.     0.   ]
#    [ 1.225  1.225]]
```

In [ ]:
def standardize(X):
    pass  # your code

In [ ]:
check(5)

**Exercise 6 · Pairwise squared distances.** $D_{ij} = \sum_d (A_{id} - B_{jd})^2$ for `A` `(m, d)` and `B` `(p, d)`. Return **shape `(m, p)`**.

**Example**: what goes in, and what should come out:
```python
A = np.array([[0.0, 0.0], [1.0, 1.0]])              # (2, 2)
B = np.array([[1.0, 0.0], [3.0, 4.0], [0.0, 0.0]])  # (3, 2)
pairwise_sq_dists(A, B)
# → [[ 1. 25.  0.]
#    [ 1. 13.  2.]]
```

In [ ]:
def pairwise_sq_dists(A, B):
    pass  # your code

In [ ]:
check(6)

**Exercise 7 · Row-by-row dot products.** $o_i = \sum_j A_{ij} B_{ij}$: the dot product of row $i$ of `A` with row $i$ of `B` (same shape `(n, d)`). Return **shape `(n,)`**. (Not `A @ B.T`: that computes all $n^2$ pairs.)

**Example**: what goes in, and what should come out:
```python
row_dots(np.array([[1.0, 2.0], [3.0, 4.0]]), np.array([[5.0, 6.0], [7.0, 8.0]]))    # → [17. 53.]   1·5 + 2·6, 3·7 + 4·8
```

In [ ]:
def row_dots(A, B):
    pass  # your code

In [ ]:
check(7)

## 6. Matrix multiplication = a sum over a shared index

$$
(XW)_{ij} = \sum_{k} X_{ik}\, W_{kj}
$$

🖼 **Picture.** One entry of `X @ W`: walk along row $i$ of $X$ and down column $j$ of $W$, multiply pairs, add. The shared index $k$ is summed away.

In [ ]:
fig_matmul_cell()

By the method: indices $i, k, j$; `X` has $(i, k)$ → `X[:, :, None]`, shape $(n, d, 1)$; `W` has $(k, j)$ → `W[None, :, :]`, shape $(1, d, h)$; multiply → $(n, d, h)$; sum over $k$ (axis 1) → $(n, h)$. That's exactly `X @ W`:

In [ ]:
X = np.array([[1, 2], [3, 4], [5, 6]])           # (3, 2)
W = np.array([[1, 0, -1], [2, 1, 0]])            # (2, 3)
by_method = (X[:, :, None] * W[None, :, :]).sum(axis=1)    # (3, 2, 3) -> sum over k -> (3, 3)
print(np.array_equal(by_method, X @ W))          # → True
print(X @ W)
# → [[ 5  2 -1]
#    [11  4 -3]
#    [17  6 -5]]

`@` gives the same numbers without building the $(n, d, h)$ array, and it runs on highly optimized code (BLAS), so it's far faster. **Rule: a sum over an index shared by two arrays is a matrix product.** Which product depends on where the shared index sits:

| Formula | NumPy | Shape | You'll meet it as |
|---|---|---|---|
| $\sum_k a_k b_k$ | `a @ b` | `()` | a dot product |
| $\sum_k A_{ik} b_k$ | `A @ b` | `(n,)` | logistic scores $Xw$ |
| $\sum_k A_{ik} B_{kj}$ | `A @ B` | `(n, h)` | a dense layer, `X @ W` |
| $\sum_i A_{ik} B_{ij}$ (shared **first** index) | `A.T @ B` | `(K, h)` | the weight gradient $X^\top G$ (§8) |
| $\sum_j A_{ij} B_{kj}$ (shared **last** index) | `A @ B.T` | `(n, K)` | all dot products between two sets of rows |

**A use: distances without the big array.** $\lVert a - b \rVert^2 = \lVert a \rVert^2 - 2\, a^\top b + \lVert b \rVert^2$, and all the $a_i^\top b_j$ at once are `A @ B.T` (the last row of the table). So the distance table needs only $(m, p)$ memory, never $(m, p, d)$. scikit-learn computes distances this way:

In [ ]:
A = np.array([[0.0, 0.0], [1.0, 1.0]]); B = np.array([[1.0, 0.0], [3.0, 4.0], [0.0, 0.0]])
D = (A ** 2).sum(axis=1)[:, None] - 2 * A @ B.T + (B ** 2).sum(axis=1)[None, :]   # (2, 1) - (2, 3) + (1, 3)
print(D)
# → [[ 1. 25.  0.]
#    [ 1. 13.  2.]]

**Exercise 8 · Cosine similarity of every pair.** $S_{ij} = \dfrac{a_i^\top b_j}{\lVert a_i \rVert\, \lVert b_j \rVert}$ for the rows $a_i$ of `A` `(m, d)` and $b_j$ of `B` `(p, d)`. Return **shape `(m, p)`**. One `@` for all the dot products; `np.linalg.norm(A, axis=1, keepdims=True)` gives the row lengths as a column.

**Example**: what goes in, and what should come out:
```python
cosine_similarity(np.array([[1.0, 0.0], [1.0, 1.0]]), np.array([[0.0, 2.0], [3.0, 3.0]]))
# → [[0.    0.707]
#    [0.707 1.   ]]     (1, 0) ⟂ (0, 2) → 0; (1, 1) and (3, 3) point the same way → 1
```

In [ ]:
def cosine_similarity(A, B):
    pass  # your code

In [ ]:
check(8)

## 7. A dense layer, forward

**The question:** a neural-network layer turns each sample's $d$ inputs into $h$ outputs. What's the formula, and the one line of NumPy for a whole batch?

$$
z_{ik} = \sum_{j=1}^{d} x_{ij}\, W_{jk} + b_k \qquad\Longleftrightarrow\qquad \underbrace{Z}_{(n,\, h)} = \underbrace{X}_{(n,\, d)}\, \underbrace{W}_{(d,\, h)} + \underbrace{b}_{(h,)}
$$

| Symbol | Shape | A row is | A column is | An entry is |
|---|---|---|---|---|
| $X$ | $(n, d)$ | one sample | one input feature | $x_{ij}$: feature $j$ of sample $i$ |
| $W$ | $(d, h)$ | one input feature | one unit (output) | $W_{jk}$: the weight from input $j$ to unit $k$ |
| $b$ | $(h,)$ |  |  | $b_k$: unit $k$'s bias |
| $Z$ | $(n, h)$ | one sample | one unit | $z_{ik}$: unit $k$'s value for sample $i$ |

🖼 **Picture.** The shapes. The input index $j$ is shared by $X$ and $W$ and is summed away (§6's rule); $b$ lines up with the last axis, so the same bias is added to every sample.

In [ ]:
fig_dense_shapes()

By hand for sample 0 below: $x_0 = (1, 0, 2)$, so unit 0 gets $1 \cdot 1 + 0 \cdot 0 + 2 \cdot 1 + 0.5 = 3.5$ and unit 1 gets $1 \cdot (-1) + 0 \cdot 2 + 2 \cdot 0 - 0.5 = -1.5$. The three-loop version and the one-liner agree:

In [ ]:
X = np.array([[1, 0, 2], [0, 1, 1], [2, 1, 0], [1, 1, 1.0]])   # (4, 3): 4 samples, 3 features
W = np.array([[1, -1], [0, 2], [1, 0.0]])                       # (3, 2): 3 inputs -> 2 units
b = np.array([0.5, -0.5])                                       # (2,)

Z_loop = np.zeros((4, 2))
for i in range(4):                  # each sample
    for k in range(2):              # each unit
        for j in range(3):          # sum over the inputs
            Z_loop[i, k] += X[i, j] * W[j, k]
        Z_loop[i, k] += b[k]
Z = X @ W + b                       # (4, 3) @ (3, 2) + (2,) -> (4, 2)
print(np.array_equal(Z, Z_loop), Z[0])     # → True [ 3.5 -1.5]

**Preview of PyTorch (`06`).** `nn.Linear(d, h)` stores its weight transposed, shape $(h, d)$ (one **row** per unit), and computes `X @ weight.T + bias`. Same numbers, different storage; keep it in mind when you read PyTorch code.

**Exercise 9 · Dense layer, forward.** $Z = XW + b$: `X` `(n, d)`, `W` `(d, h)`, `b` `(h,)`. Return **shape `(n, h)`**.

**Example**: what goes in, and what should come out:
```python
dense_forward(X, W, b)     # X, W, b from the cell above
# → [[ 3.5 -1.5]
#    [ 1.5  1.5]
#    [ 2.5 -0.5]
#    [ 2.5  0.5]]
```

In [ ]:
def dense_forward(X, W, b):
    pass  # your code

In [ ]:
check(9)

## 8. A dense layer, backward: the shapes decide

**The question:** to train the layer we need the gradient of the loss $L$ (one number) with respect to $W$, $b$ and $X$. Suppose someone hands us $G = \frac{\partial L}{\partial Z}$, the gradient with respect to the layer's output, shape $(n, h)$, entry $G_{ik} = \partial L / \partial z_{ik}$. (`03` shows where $G$ comes from.)

**Proposition.** For $Z = XW + b$ and $G = \partial L / \partial Z$:

$$
\underbrace{\frac{\partial L}{\partial W}}_{(d,\, h)} = X^\top G, \qquad \underbrace{\frac{\partial L}{\partial b}}_{(h,)} = \sum_{i=1}^{n} G_{i,:}, \qquad \underbrace{\frac{\partial L}{\partial X}}_{(n,\, d)} = G\, W^\top
$$

**Proof.** Write the layer for one entry: $z_{ik} = \sum_j x_{ij} W_{jk} + b_k$. The weight $W_{jk}$ appears in $z_{ik}$ for **every** sample $i$ (and only in column $k$), with $\partial z_{ik} / \partial W_{jk} = x_{ij}$. By the chain rule, summing over everything $W_{jk}$ affects:

$$
\begin{aligned} \frac{\partial L}{\partial W_{jk}} &= \sum_{i} \frac{\partial L}{\partial z_{ik}}\, \frac{\partial z_{ik}}{\partial W_{jk}} = \sum_i G_{ik}\, x_{ij} = \sum_i (X^\top)_{ji}\, G_{ik} = (X^\top G)_{jk}, \\ \frac{\partial L}{\partial b_k} &= \sum_i G_{ik} \cdot \frac{\partial z_{ik}}{\partial b_k} = \sum_i G_{ik} \cdot 1, \\ \frac{\partial L}{\partial x_{ij}} &= \sum_k \frac{\partial L}{\partial z_{ik}}\, \frac{\partial z_{ik}}{\partial x_{ij}} = \sum_k G_{ik}\, W_{jk} = (G W^\top)_{ij}. \qquad \blacksquare \end{aligned}
$$

**Remarks.** (1) **The shapes decide.** $\partial L / \partial W$ must have $W$'s shape $(d, h)$, and from $X$ $(n, d)$ and $G$ $(n, h)$ the only product with that shape is $X^\top G$; the same argument gives $G W^\top$ for $X$'s shape. If you forget the formula in the exam, rebuild it from the shapes. (2) **The sum over $i$**: every sample adds its own contribution to the same weight. If $L$ is a mean over the batch, the $\frac{1}{n}$ is already inside $G$. (3) **A copy becomes a sum.** In the forward pass $b$ is copied to every row; in the backward pass the rows of $G$ are added up. (4) $\partial L / \partial X$ isn't used to update anything; it's passed back to the previous layer (`03`).

🖼 **Picture.** The three gradients as shapes. Each one has the shape of the thing it's the gradient of.

In [ ]:
fig_dense_backward_shapes()

**Example, checked two ways.** Take $L = \sum_{i,k} C_{ik}\, z_{ik}$ for a fixed table $C$. Then $\partial L / \partial z_{ik} = C_{ik}$, so $G = C$. The formula gives $X^\top C$; nudging each weight by $\pm 10^{-6}$ and measuring the change in $L$ (a numerical derivative) gives the same:

In [ ]:
C = np.array([[1, 0], [0, 1], [1, 1], [-1, 2.0]])     # (4, 2), so G = C
L = lambda W_: ((X @ W_ + b) * C).sum()                # the loss as a function of W

dW_formula = X.T @ C                                    # (3, 4) @ (4, 2) -> (3, 2)
dW_numeric = np.zeros_like(W)
for j in range(3):
    for k in range(2):
        E = np.zeros_like(W); E[j, k] = 1e-6            # nudge one weight
        dW_numeric[j, k] = (L(W + E) - L(W - E)) / 2e-6
print(dW_formula.ravel(), np.allclose(dW_formula, dW_numeric))   # → [2. 4. 0. 4. 1. 3.] True
print(C.sum(axis=0), C @ W.T)
# → [1. 4.] [[ 1.  0.  1.]
#    [-1.  2.  0.]
#    [ 0.  2.  1.]
#    [-3.  4. -1.]]

**Recipe (any layer's backward pass).** Write the forward formula for one entry; find every place a parameter appears; chain rule, summing over those places; then translate the sum into NumPy with the method (or let the shapes pick the product).

**Exercise 10 · Dense layer, backward.** Return `(dX, dW, db)` for $Z = XW + b$ given $G = \partial L / \partial Z$: $dX = G W^\top$ `(n, d)`, $dW = X^\top G$ `(d, h)`, $db = $ the column sums of $G$, `(h,)`.

**Example**: what goes in, and what should come out:
```python
dX, dW, db = dense_backward(X, W, C)     # X, W, C from the cells above
dW    # → [[2. 4.]  [0. 4.]  [1. 3.]]
db    # → [1. 4.]
dX    # → [[ 1.  0.  1.]  [-1.  2.  0.]  [ 0.  2.  1.]  [-3.  4. -1.]]
```

In [ ]:
def dense_backward(X, W, G):
    pass  # your code

In [ ]:
check(10)

## 9. Softmax and cross-entropy, row by row

**The question:** a classifier's last layer gives each sample one score per class, the **logits** $Z$, shape $(n, C)$. How do they become probabilities, and how is the loss computed for a whole batch?

$$
p_{ic} = \frac{e^{z_{ic}}}{\sum_{c'} e^{z_{ic'}}}, \qquad L = -\frac{1}{n}\sum_{i=1}^{n} \log p_{i,\, y_i}
$$

| Symbol | Shape | What it is |
|---|---|---|
| $Z$ | $(n, C)$ | logits: row $i$ = sample $i$'s raw score for each class |
| $P$ | $(n, C)$ | probabilities: each row is positive and sums to 1 |
| $y$ | $(n,)$ | integer labels $0, \dots, C - 1$: the true class of each sample |
| $p_{i, y_i}$ | one number per row | the probability the model gives the true class |

**Why subtract the row maximum.** `np.exp(1000)` overflows to `inf`, and `inf / inf` is `nan`. Subtracting any constant $c$ from a row changes nothing, because $\frac{e^{z_c - c}}{\sum e^{z_{c'} - c}} = \frac{e^{-c}\, e^{z_c}}{e^{-c} \sum e^{z_{c'}}}$ and $e^{-c}$ cancels. With $c$ = the row maximum, every exponent is $\le 0$, so nothing overflows:

In [ ]:
z = np.array([1000.0, 1001.0])
print(np.exp(z) / np.exp(z).sum())                       # → [nan nan]   (overflow; NumPy also warns)
print(np.exp(z - z.max()) / np.exp(z - z.max()).sum())   # → [0.26894142 0.73105858]

**Picking the true class's probability in every row** is fancy indexing with two index arrays, `P[np.arange(n), y]`:

🖼 **Picture.** `np.arange(4)` supplies the row numbers and `y` the column numbers; they're paired position by position. The result is one probability per sample, and the loss is the mean of their $-\log$.

In [ ]:
fig_pick_entries()

With one-hot labels $Y$ `(n, C)` the same loss is $-\frac{1}{n}\sum_i \sum_c Y_{ic} \log p_{ic}$: every term with $Y_{ic} = 0$ vanishes, leaving exactly the picked entries. Both versions:

In [ ]:
P = np.array([[0.7, 0.2, 0.1], [0.1, 0.8, 0.1]]); y = np.array([0, 1])
Y = np.eye(3)[y]                                                       # one-hot, (2, 3)
print(-np.log(P[np.arange(2), y]).mean(), -(Y * np.log(P)).sum(axis=1).mean())   # → 0.2899092476264711 0.2899092476264711

**Exercise 11 · One-hot.** Row $i$ is all zeros except a 1 in column $y_i$. `y` is `(n,)` integers in `0 .. C-1`. Return **shape `(n, C)`**, floats.

**Example**: what goes in, and what should come out:
```python
one_hot(np.array([2, 0, 1]), 3)
# → [[0. 0. 1.]
#    [1. 0. 0.]
#    [0. 1. 0.]]
```

In [ ]:
def one_hot(y, C):
    pass  # your code

In [ ]:
check(11)

**Exercise 12 · Cross-entropy.** $L = -\frac{1}{n}\sum_i \log p_{i, y_i}$ for probabilities `P` `(n, C)` and integer labels `y` `(n,)`. Return a number.

**Example**: what goes in, and what should come out:
```python
cross_entropy(np.array([[0.7, 0.2, 0.1], [0.1, 0.8, 0.1]]), np.array([0, 1]))    # → 0.2899   (-(log 0.7 + log 0.8) / 2)
```

In [ ]:
def cross_entropy(P, y):
    pass  # your code

In [ ]:
check(12)

## 10. einsum: the method in one string (optional)

`np.einsum` takes the indices directly: name the indices of each input, then of the output; **every index missing from the output is summed**. It's the method of §2 written down literally, and it appears in PyTorch code (`torch.einsum`, e.g. in attention, Week 8).

| Formula | einsum | Same as |
|---|---|---|
| $\sum_i r_{ik} x_i$ | `np.einsum("ik,i->k", R, x)` | `(R * x[:, None]).sum(axis=0)` |
| $\sum_k X_{ik} W_{kj}$ | `np.einsum("ik,kj->ij", X, W)` | `X @ W` |
| $\sum_i X_{ij} G_{ik}$ | `np.einsum("ij,ik->jk", X, G)` | `X.T @ G` |
| $\sum_j A_{ij} B_{ij}$ | `np.einsum("ij,ij->i", A, B)` | `(A * B).sum(axis=1)` |
| $a_i b_k$ | `np.einsum("i,k->ik", a, b)` | `a[:, None] * b[None, :]` |

In [ ]:
print(np.einsum("ik,i->k", R, x).round(3), (R * x[:, None]).sum(axis=0).round(3))   # → [ 5.69 21.31] [ 5.69 21.31]
print(np.array_equal(np.einsum("ij,ik->jk", X, C), X.T @ C))                     # → True

## 11. The recipe

**Formula → NumPy, on one card.**

- Write one entry of the result, every index visible.
- Anatomy table: symbol → indices → array → shape. The result's indices, in order, are its axes.
- Line up every input with `None` where it lacks an index (optional only for an index on the last axis).
- Combine elementwise. Shape comment on every line.
- Sum every index that is inside but not in the result. A sum over an index shared by two arrays is a matrix product (§6's table).
- Check the shape and one entry by hand.
- Stuck? Write the loop, find each line in §5's table.

Run all the checks:

In [ ]:
for n in range(1, 13):
    check(n)

---
Solutions: `solutions/01_solutions.py`. Next: `02_neural_networks.ipynb`, where these lines become a network.